# Imports

In [1]:
import warnings
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
FOLDS = 5
ID = 'id'
TARGET = 'class'
TARGET_MAPPING = {
    "GALAXY": 0,
    "QSO": 1,
    "STAR": 2
}
TARGET_INV_MAPPING = {
    0:"GALAXY",
    1:"QSO",
    2:"STAR"
}
RUN_OPTUNA = False
RUN_SKF = True

# Data Loading & Original Dataset Preparation

In [2]:
train = pd.read_csv("playground-series-s6e6\\train.csv")
test = pd.read_csv("playground-series-s6e6\\test.csv")
original = pd.read_csv("playground-series-s6e6\\externel_dataset\\star_classification.csv") 

train_id = train[ID]
test_id = test[ID]

def get_spectral_type(g, r):
    return pd.cut(
        r - g, 
        [-np.inf, -1, -0.5, 0, np.inf],
        labels=['M', 'G/K', 'A/F', 'O/B']
    ).astype(str)

def get_galaxy_population(u, r):
    return pd.cut(
        u - r, 
        [-np.inf, 1.4, 2.2, np.inf],
        labels=['Blue_Cloud', 'Green_Valley', 'Red_Sequence']
    ).astype(str)

original['spectral_type'] = get_spectral_type(original['g'], original['r'])
original['galaxy_population'] = get_galaxy_population(original['u'], original['r'])

target_map = {'GALAXY': 'GALAXY', 'QSO': 'QSO', 'STAR': 'STAR'}
original[TARGET] = original[TARGET].map(target_map)
train[TARGET] = train[TARGET].map(target_map)

base_features = [col for col in train.columns if col not in [TARGET, ID]]
cat_cols_init = train.drop(columns=[ID, TARGET]).select_dtypes(include=['object']).columns.tolist()
num_cols_init = [c for c in train.drop(columns=[ID, TARGET]).select_dtypes(exclude=['object']).columns]

original_numeric_target = original[TARGET].map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})
original_global_mean = original_numeric_target.mean()
original_global_median = original_numeric_target.median()

original_stats = {}
for col in base_features:
    if col in original.columns:
        orig_temp = original[[col]].copy()
        orig_temp[TARGET] = original_numeric_target
        if col in num_cols_init:
            orig_temp[col] = np.floor(orig_temp[col])
            
        stats = orig_temp.groupby(col)[TARGET].agg(['mean', 'median', 'std', 'skew', 'count']).reset_index()
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ['mean', 'median', 'std', 'skew', 'count']]
        original_stats[col] = stats

# Feature Engineering Pipeline Definition

In [3]:
# %% [code]
from __future__ import annotations
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.base import BaseEstimator, TransformerMixin
from pandas.api.types import is_numeric_dtype

from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple


class FeatureFactory(BaseEstimator, TransformerMixin):
    """
    Feature engineering pipeline for PS-S06E06: Predicting Stellar Class.

    Target classes: GALAXY (2), QSO (0), STAR (1)
    Raw features: alpha, delta, u, g, r, i, z, redshift, spectral_type, galaxy_population
    """

    valid_strategies = [
        'encoding',
        'colors',             # Photometric color indices (magnitude differences)
        'ratios',             # Flux/magnitude ratios and spectral shape features
        'interactions',       # Cross-feature interaction terms
        'redshift',           # Redshift-derived cosmological features
        'position',           # Sky-position features
        'flux',               # Derived flux features
        'numeric_expansion',  # Log, sqrt, ratios, and differences
    ]

    def __init__(
        self,
        *,
        strategies=None,
        seed=10301,
        target='class',
        verbose=False
    ):
        if strategies is None:
            strategies = []

        self.strategies = []
        self.seed = seed
        self.target = target
        self.verbose = verbose
        self._engineered_cat_cols_: List[str] = []

        invalid_strategies = set()
        for strategy in strategies:
            if strategy in self.valid_strategies:
                self.strategies.append(strategy)
            else:
                invalid_strategies.add(strategy)

        if invalid_strategies:
            raise ValueError(
                f'Invalid FeatureFactory strategies requested: {",".join(invalid_strategies)}'
            )

        # Categorical columns present in the raw data
        self.cat_cols = ['spectral_type', 'galaxy_population']

        # Learned state (set in fit)
        self._is_fit: bool = False

    # ----------------------------
    # Public API
    # ----------------------------
    def fit(self, df: pd.DataFrame = None) -> 'FeatureFactory':
        if self.verbose:
            print('  -> Fitting DataFrame...')
            
        # Always drop id and target
        df.drop('id', axis=1, inplace=True, errors='ignore')
        df.drop(self.target, axis=1, inplace=True, errors='ignore')
        
        self.num_features_ = df.select_dtypes(exclude=['object', 'bool', 'category']).columns.tolist()
        self.cat_features_ = df.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()

        self._is_fit = True
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        if not self._is_fit:
            raise RuntimeError('FeatureFactory must be fit() before transform().')

        if self.verbose:
            print(f'Applying FeatureFactory with strategies: {", ".join(self.strategies)}')

        df_new = df.copy()
            
        # Strategy order matters: colors first (they feed ratios/interactions)
        if 'colors' in self.strategies:
            df_new = self._add_colors(df_new)

        if 'ratios' in self.strategies:
            df_new = self._add_ratios(df_new)

        if 'redshift' in self.strategies:
            df_new = self._add_redshift_features(df_new)

        if 'position' in self.strategies:
            df_new = self._add_position_features(df_new)

        if 'interactions' in self.strategies:
            df_new = self._add_interactions(df_new)

        if 'encoding' in self.strategies:
            df_new = self._add_encoding(df_new)

        if 'flux' in self.strategies:
            df_new = self._add_flux(df_new)

        if 'numeric_expansion' in self.strategies:
            df_new = self._add_numeric_expansion(df_new)

        if self.target in df_new.columns:
            df_new = df_new.drop(self.target, axis=1)

        # If any feature has just one value, it's just noise.
        drop = [c for c in df_new.columns if df_new[c].nunique(dropna=False) == 1]
        if drop:
            df_new.drop(drop, axis=1, inplace=True)

        return df_new

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)

    def get_strategies(self) -> list:
        return self.strategies


    # -------------------------
    # Internals
    # -------------------------
    def _add_encoding(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Ordinal-encode the two categorical features so XGBoost native-cat
        mode can consume them, or so they survive a category→int fallback.
        The columns are cast to pandas Categorical dtype; XGBoost with
        enable_categorical=True will handle them natively.
        """
        if self.verbose:
            print('  -> Adding encoding features...')

        # spectral_type: O/B < A/F < G/K < M  (rough temperature order)
        spectral_order = ['O/B', 'A/F', 'G/K', 'M']
        if 'spectral_type' in df.columns:
            df['spectral_type'] = pd.Categorical(
                df['spectral_type'], categories=spectral_order, ordered=True
            )

        # galaxy_population: no strong intrinsic order → unordered categorical
        if 'galaxy_population' in df.columns:
            df['galaxy_population'] = df['galaxy_population'].astype('category')

        return df

    
    def _add_colors(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Standard SDSS photometric color indices (magnitude differences).
        These are the primary discriminators between GALAXY, QSO, and STAR
        in the Sloan Digital Sky Survey color-color space.

        Bands ordered by wavelength: u < g < r < i < z
        """
        if self.verbose:
            print('  -> Adding photometric color features...')

        bands = ['u', 'g', 'r', 'i', 'z']
        # Consecutive colors
        df['u_minus_g'] = df['u'] - df['g']
        df['g_minus_r'] = df['g'] - df['r']
        df['r_minus_i'] = df['r'] - df['i']
        df['i_minus_z'] = df['i'] - df['z']

        # Wider-baseline colors (strong QSO vs STAR discriminators)
        df['u_minus_r'] = df['u'] - df['r']
        df['u_minus_z'] = df['u'] - df['z']
        df['g_minus_z'] = df['g'] - df['z']
        df['g_minus_i'] = df['g'] - df['i']

        # Total color span across all bands
        df['color_range'] = df['u'] - df['z']

        return df

    
    def _add_ratios(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Magnitude-based ratio and spectral-shape features.
        Because SDSS magnitudes are on an AB system (log-flux), differences
        are equivalent to flux ratios in log-space; we also derive explicit
        curvature proxies.
        """
        if self.verbose:
            print('  -> Adding ratio/spectral-shape features...')

        # Spectral slope proxies (blue vs red)
        if 'u_minus_g' in df.columns and 'r_minus_i' in df.columns:
            df['blue_red_slope'] = df['u_minus_g'] - df['r_minus_i']

        # color curvature (concavity of the SED in g-r-i)
        if all(c in df.columns for c in ['g_minus_r', 'r_minus_i']):
            df['color_curvature'] = df['g_minus_r'] - df['r_minus_i']

        # Mean magnitude (overall brightness)
        bands = ['u', 'g', 'r', 'i', 'z']
        df['mean_mag'] = df[bands].mean(axis=1)

        # Standard deviation across bands (SED flatness)
        df['std_mag'] = df[bands].std(axis=1)

        # Ratio of outer bands to central band (SED shape)
        df['outer_to_r'] = (df['u'] + df['z']) / (2.0 * df['r'] + 1e-5)

        return df

    
    def _add_redshift_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Features derived from redshift.  Redshift is the single strongest
        discriminator in this dataset:
          - STARs:    z ≈ 0
          - GALAXYs:  z ~ 0.1–1
          - QSOs:     z ~ 0.1–5+

        We also interact redshift with photometric colors because the
        4000-Å break shifts through different SDSS bands at different redshifts.
        """
        if self.verbose:
            print('  -> Adding redshift features...')

        z = df['redshift']

        df['redshift_log1p'] = np.log1p(np.clip(z, 0, None))
        df['redshift_sq']    = z ** 2
        df['is_near_zero_z'] = (np.abs(z) < 0.004).astype(np.int8)   # strong STAR flag
        df['is_high_z']      = (z > 1.0).astype(np.int8)             # QSO-dominated region

        # Redshift × color interactions
        if 'u_minus_g' in df.columns:
            df['z_x_u_g'] = z * df['u_minus_g']
        if 'g_minus_r' in df.columns:
            df['z_x_g_r'] = z * df['g_minus_r']
        if 'color_range' in df.columns:
            df['z_x_color_range'] = z * df['color_range']

        df['is_mid_z'] = ((z >= 0.1) & (z <= 0.5)).astype(np.int8)

        return df

    
    def _add_position_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Sky-coordinate features from right ascension (alpha) and
        declination (delta).  The Galactic plane is a natural boundary
        between STAR-rich and extragalactic-rich sky regions.
        """
        if self.verbose:
            print('  -> Adding sky-position features...')

        alpha_rad = np.deg2rad(df['alpha'])
        delta_rad = np.deg2rad(df['delta'])

        # Cartesian projection onto the unit sphere
        df['pos_x'] = np.cos(delta_rad) * np.cos(alpha_rad)
        df['pos_y'] = np.cos(delta_rad) * np.sin(alpha_rad)
        df['pos_z'] = np.sin(delta_rad)

        # Galactic latitude proxy: SDSS avoids |b| < ~30°, but small
        # residual latitude effects remain
        df['abs_delta'] = np.abs(df['delta'])

        df['alpha_sin'] = np.sin(alpha_rad)
        df['alpha_cos'] = np.cos(alpha_rad)
        df['delta_sin'] = np.sin(delta_rad)
        df['delta_cos'] = np.cos(delta_rad)
    
        df['alpha_bin'] = pd.cut(df['alpha'], bins=24, labels=False, include_lowest=True)
        df['delta_bin'] = pd.cut(df['delta'], bins=18, labels=False, include_lowest=True)
        df['sky_bin'] = df['alpha_bin'].astype(str) + '_' + df['delta_bin'].astype(str)

        return df

        
    def _add_interactions(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Higher-order interactions between the most discriminating features.
        """
        if self.verbose:
            print('  -> Adding interaction features...')

        z = df['redshift']

        # Redshift × brightness
        df['z_x_mean_mag'] = z * df['mean_mag'] if 'mean_mag' in df.columns else z * df['r']

        # color-color products (analogous to color-color diagram axes)
        if 'u_minus_g' in df.columns and 'g_minus_r' in df.columns:
            df['ug_x_gr'] = df['u_minus_g'] * df['g_minus_r']

        if 'g_minus_r' in df.columns and 'r_minus_i' in df.columns:
            df['gr_x_ri'] = df['g_minus_r'] * df['r_minus_i']

        # SED flatness × redshift
        if 'std_mag' in df.columns:
            df['std_mag_x_z'] = df['std_mag'] * z

        if 'u_minus_g' in df.columns and 'i_minus_z' in df.columns and 'std_mag' in df.columns:
            df['color_skew'] = (df['u_minus_g'] - df['i_minus_z']) / (df['std_mag'] + 1e-5)
        
        return df

    
    def _add_flux(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Flux-derived features.
        """
        if self.verbose:
            print('  -> Adding flux features...')
            
        # Band pair ratios in linear flux space
        for band in ['u', 'g', 'r', 'i', 'z']:
            df[f'flux_{band}'] = np.power(10, -0.4 * df[band])
        df['flux_u_over_g'] = df['flux_u'] / (df['flux_g'] + 1e-9)
        df['total_flux']    = df[['flux_u','flux_g','flux_r','flux_i','flux_z']].sum(axis=1)

        return df

    
    def _add_numeric_expansion(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Add square, square root, log, ratios, and differences 
        of all of the original numeric features
        """
        if self.verbose:
            print('  -> Adding numeric_expansion features...')

        for c in self.num_features_:
            df[f"Log_{c}"] = np.log1p(df[c])
            df[f"{c}_sq"] = df[c]**2            
            df[f"{c}_sqrt"] = df[c]**0.5
        
        return df

        
    # -------------------------
    # Utility
    # -------------------------
    def get_cat_features(self, df: pd.DataFrame) -> List[str]:
        """
        Returns the list of categorical columns present in df, including
        any engineered categorical columns tracked by this instance.
        """
        cat_cols = []
        for col in df.columns:
            if df[col].dtype == 'object' or str(df[col].dtype).startswith('category'):
                cat_cols.append(col)

        for c in self._engineered_cat_cols_:
            if c in df.columns and c not in cat_cols:
                cat_cols.append(c)

        # Stable deduplicated order
        seen: set = set()
        out: List[str] = []
        for c in cat_cols:
            if c not in seen:
                out.append(c)
                seen.add(c)
        return out

# Feature Engineering Execution 

In [ ]:
X_train = train.drop(columns=[ID, TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[ID])

chosen_strategies = ['encoding', 'colors', 'ratios', 'redshift', 'position', 'interactions', 'flux']

factory = FeatureFactory(
    strategies=chosen_strategies,
    target='class',
    verbose=True
)

print("Processing Training Data...")
X_train = factory.fit_transform(X_train)
print("\nProcessing Testing Data...")
X_test = factory.transform(X_test)

SKEW_THRESHOLD = 1.5
UNIQUENESS_THRESHOLD = 15
current_num_cols = X_train.select_dtypes(exclude=['category', 'object']).columns.tolist()

for col in current_num_cols:
    if X_train[col].nunique() <= UNIQUENESS_THRESHOLD: continue
    if abs(X_train[col].skew()) > SKEW_THRESHOLD:
        X_train[col] = (np.sign(X_train[col]) * np.log1p(np.abs(X_train[col]))).astype('float32')
        X_test[col] = (np.sign(X_test[col]) * np.log1p(np.abs(X_test[col]))).astype('float32')

for col in X_train.columns.tolist():
    if isinstance(X_train[col].dtype, pd.CategoricalDtype) or X_train[col].dtype == 'object' or X_train[col].nunique() <= UNIQUENESS_THRESHOLD:
        X_train[col], X_test[col] = pd.Categorical(X_train[col]), pd.Categorical(X_test[col])

constant_cols = [col for col in X_train.columns if X_train[col].nunique() <= 1]
if constant_cols:
    X_train.drop(columns=constant_cols, inplace=True)
    X_test.drop(columns=constant_cols, inplace=True)

Processing Training Data...
  -> Fitting DataFrame...
Applying FeatureFactory with strategies: encoding, colors, ratios, redshift, position, interactions, flux
  -> Adding photometric color features...
  -> Adding ratio/spectral-shape features...
  -> Adding redshift features...
  -> Adding sky-position features...
  -> Adding interaction features...
  -> Adding encoding features...
  -> Adding flux features...

Processing Testing Data...
Applying FeatureFactory with strategies: encoding, colors, ratios, redshift, position, interactions, flux
  -> Adding photometric color features...
  -> Adding ratio/spectral-shape features...
  -> Adding redshift features...
  -> Adding sky-position features...
  -> Adding interaction features...
  -> Adding encoding features...
  -> Adding flux features...


## Data Processing

In [16]:
from sklearn.preprocessing import OrdinalEncoder

def ordinal_encode(train_df, test_df=None):
    train_df = train_df.copy()

    cat_cols = train_df.select_dtypes(include=['object', bool, 'category']).columns.tolist()

    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )

    if test_df is not None:
        test_df = test_df.copy()

        encoder.fit(
            pd.concat(
                [train_df[cat_cols], test_df[cat_cols]],
                axis=0
            )
        )

        train_df[cat_cols] = encoder.transform(train_df[cat_cols])
        test_df[cat_cols] = encoder.transform(test_df[cat_cols])

        return train_df, test_df, encoder

    encoder.fit(train_df[cat_cols])
    train_df[cat_cols] = encoder.transform(train_df[cat_cols])

    return train_df, encoder


train_df_enc, test_df_enc, encoder = ordinal_encode(X_train, X_test)
y_train = y_train.map(TARGET_MAPPING)

In [19]:
# train_df_enc.to_csv("processed_data\\processed_train_new_fe.csv", index=False)
# test_df_enc.to_csv("processed_data\\processed_test_new_fe.csv", index=False)
# y_train.to_csv("processed_data\\processed_target_new_fe.csv", index=False)

## Optuna Search

In [20]:
import lightgbm as lgb
import optuna
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

print(lgb.__version__)

X_train_, X_test_, y_train_, y_test_ = train_test_split(train_df_enc, y_train, test_size=0.2, random_state=42, stratify=y_train)

lgb_best_params = {
    'n_estimators': 1841,
    'learning_rate': 0.016121467631636684,
    'num_leaves': 126,
    'max_depth': 12,
    'min_child_samples': 7,
    'subsample': 0.9137011411523339,
    'colsample_bytree': 0.6472035788104408,
    'reg_alpha': 1.6143965568012781,
    'reg_lambda': 0.0457226857453295,
    'min_split_gain': 0.520062046419683,
    "max_bin": 63
}

def objective(trial):

    params = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "device": "gpu",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            max(1200, int(lgb_best_params["n_estimators"] * 0.8)),
            min(2500, int(lgb_best_params["n_estimators"] * 1.2))
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            lgb_best_params["learning_rate"] * 0.7,
            lgb_best_params["learning_rate"] * 1.3,
            log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            max(31, lgb_best_params["num_leaves"] - 20),
            min(255, lgb_best_params["num_leaves"] + 20)
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            max(6, lgb_best_params["max_depth"] - 2),
            lgb_best_params["max_depth"] + 2
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            max(2, lgb_best_params["min_child_samples"] - 3),
            lgb_best_params["min_child_samples"] + 3
        ),

        "subsample": trial.suggest_float(
            "subsample",
            max(0.7, lgb_best_params["subsample"] - 0.10),
            min(1.0, lgb_best_params["subsample"] + 0.05)
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            max(0.4, lgb_best_params["colsample_bytree"] - 0.10),
            min(1.0, lgb_best_params["colsample_bytree"] + 0.10)
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            max(1e-4, lgb_best_params["reg_alpha"] * 0.3),
            lgb_best_params["reg_alpha"] * 3,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            max(1e-5, lgb_best_params["reg_lambda"] * 0.2),
            lgb_best_params["reg_lambda"] * 5,
            log=True
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            max(0.0, lgb_best_params["min_split_gain"] - 0.30),
            lgb_best_params["min_split_gain"] + 0.30
        ),

        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
        "max_bin": 63
    }

    model = lgb.LGBMClassifier(**params)

    model.fit(X_train_, y_train_)
    y_pred = model.predict(X_test_)
    b_acc = balanced_accuracy_score(y_test_, y_pred)
    print(f"Trial {trial.number} | BAL_ACC: {b_acc:.6f}")

    return b_acc


def best_trial_callback(study, trial):

    print("\n" + "=" * 60)
    print(f"Trial {trial.number} Completed")
    print(f"Trial BAL_ACC: {trial.value:.6f}")
    print(f"\nBest BAL_ACC So Far: {study.best_value:.6f}")
    print("\nBest Parameters So Far:")
    print(study.best_params)
    print("=" * 60)


if RUN_OPTUNA:
    sampler = optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=5,
        multivariate=True,
        group=True
    )

    study_lgb = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="lightgbm_local_search"
    )

    study_lgb.enqueue_trial(lgb_best_params)

    study_lgb.optimize(
        objective,
        n_trials=100,
        callbacks=[best_trial_callback],
        show_progress_bar=True
    )

    print("\nBest BAL_ACC:")
    print(study_lgb.best_value)

    print("\nBest Parameters:")
    print(study_lgb.best_params)

4.6.0


## Best LightGBM 

In [30]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import lightgbm as lgb

SEED = 42
RUN_SKF = False
N_FOLDS = 5

best_lgb_params = {'n_estimators': 1402, 'learning_rate': 0.015683531088405856, 'num_leaves': 138, 'max_depth': 15, 'min_child_samples': 9, 'subsample': 0.900228907109458, 'colsample_bytree': 0.6185476387342294, 'reg_alpha': 7.804739536508066, 'reg_lambda': 0.06876460724759273, 'min_split_gain': 0.08356013475510977}

best_lgb_params['class_weight']='balanced'
best_lgb_params['random_state']=42
best_lgb_params['max_bin']=63
best_lgb_params['objective']="multiclass"
best_lgb_params["metric"] = "multi_logloss"

if RUN_SKF:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_scores = []

    print(f"{'Fold':<6} {'Balanced Accuracy':^20}")
    print("-" * 30)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_, y_train_), 1):
        X_tr, X_val = X_train_.iloc[train_idx], X_train_.iloc[val_idx]
        y_tr, y_val = y_train_.iloc[train_idx], y_train_.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
        **best_lgb_params,
        verbose=-1
        )
        
        model.fit(X_tr, y_tr)
        
        y_pred = model.predict(X_val)
        balanced_acc = balanced_accuracy_score(y_val, y_pred)
        fold_scores.append(balanced_acc)
        
        print(f"{fold:<6} {balanced_acc:>20.4f}")

    print("-" * 30)
    print(f"{'Mean':<6} {np.mean(fold_scores):>20.4f}")
    print(f"{'Std':<6} {np.std(fold_scores):>20.4f}")

## Submission

In [31]:
lgb_model = lgb.LGBMClassifier(
    **best_lgb_params, 
    verbose=-1
    )
lgb_model.fit(train_df_enc, y_train)

FILE_NAME = "lgb_19_06_2026_new_features_v1"
SUB_FILE_NAME = f"submissions\\{FILE_NAME}.csv"
PROB_FILE_NAME = f"probabilities\\{FILE_NAME}.csv"

probabilities = lgb_model.predict_proba(test_df_enc)
class_names = ['GALAXY', 'QSO', 'STAR'] 
proba_df_classes = pd.DataFrame(probabilities, columns=class_names)
proba_df = pd.concat([test[ID].reset_index(drop=True), proba_df_classes], axis=1)
proba_df.to_csv(PROB_FILE_NAME, index=False)

predictions = pd.Series(lgb_model.predict(test_df_enc)).map(TARGET_INV_MAPPING)
sub_df = pd.DataFrame({ID:test[ID], TARGET:predictions})
sub_df.to_csv(SUB_FILE_NAME, index=False)
sub_df.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [ ]:
# LightGBM BENCHMARK
# Fold    Balanced Accuracy  
# ------------------------------
# 1                    0.9632
# 2                    0.9640
# 3                    0.9628
# 4                    0.9628
# 5                    0.9631
# ------------------------------
# Mean                 0.9632
# Std                  0.0005